# S25 · Pick the right test (t-test, chi-square, ANOVA)

A gentle tour of the three tests you will reach for again and again. For each one we
make the data ourselves so we know the true answer, run the test in a single line,
and read the result in plain words. The whole skill today is choosing the right test
for your data; the library does the arithmetic.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to the idea of a p-value? Open the primer
  `primers/p_values_in_plain_words.md` for a ten-minute, picture-first version.
- New to averages and spread? Open `primers/what_is_a_distribution.md`.
- Already confident with statistics or code? Skip to the cell marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scipy.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

## The one idea behind every test

Suppose we see a difference between two groups: version B of an app screen looks a
little better than version A. The honest question is always the same: **is that
difference real, or could pure luck have produced it?**

Every test below answers that with two competing claims:

- **Null hypothesis (H0):** there is NO real difference; any gap is chance.
- **Alternative (H1):** there IS a real difference.

Each test hands us a **p-value**: *if the null were true, how often would we see data
this extreme, or more?* A small p-value (we will use the usual line of 0.05) means the
data would be surprising if nothing were going on, so we start to doubt the null.

One warning to keep all day: the p-value is NOT the probability that the null is true.
We compute it by *assuming* the null is true, so it cannot also be the probability of
the null.

## Step 1 - import our tools

`scipy.stats` is the standard statistics library. It has a ready-made function for
every test we need, so we never write the formulas ourselves.

In [ ]:
import numpy as np                 # fast maths on lists of numbers
import matplotlib.pyplot as plt     # drawing charts
from scipy import stats             # all our hypothesis tests live in here

# Set a seed so we all get the same random numbers, and the same answers, every time.
np.random.seed(0)

print("Tools ready.")

## Step 2 - a small helper to read a p-value in plain words

We will reuse this after every test. It compares the p-value to our chosen line of
0.05 and prints a sentence anyone in the room can understand.

In [ ]:
# The significance level: our line in the sand. 0.05 is the common convention.
# We choose it BEFORE looking at the data, never after.
significance_level = 0.05

def explain_p_value(p_value):
    # Print the number first.
    print("p-value =", round(p_value, 4))
    # Then say what it means in plain words.
    if p_value < significance_level:
        print("This is SMALL (below 0.05).")
        print("The data would be surprising if there were no real difference,")
        print("so we have evidence of a real difference. We REJECT the null.")
    else:
        print("This is NOT small (0.05 or above).")
        print("The data is unremarkable if there were no real difference,")
        print("so we do NOT have strong evidence. We do NOT reject the null.")

print("Helper ready. We will call explain_p_value(...) after each test.")

## Test 1 - two-sample t-test (compare two AVERAGES)

Use this when your metric is a **number you average per user**, such as minutes spent
on the checkout page. We compare the average of group A with the average of group B.

### Step 3 - make data for two groups

We invent 'time on the checkout page' for two screen designs. We build group B to be a
little higher on average, so we know the true answer in advance: there really is a
difference. Each group is drawn from a bell-shaped spread with a known centre.

In [ ]:
# 120 users in each group. We draw their "time on page" from a normal
# distribution (a bell curve) with a chosen average and spread.
group_A_times = np.random.normal(11.0, 2.6, size=120)   # average 11.0 minutes
group_B_times = np.random.normal(12.6, 2.6, size=120)   # average 12.6 minutes

# Look at the averages we actually got.
print("group A average:", round(group_A_times.mean(), 2), "minutes")
print("group B average:", round(group_B_times.mean(), 2), "minutes")
print("difference     :", round(group_B_times.mean() - group_A_times.mean(), 2), "minutes")

### Step 4 - always look at the data first

A box plot shows the middle and the spread of each group side by side. Look before you
test: your eyes catch things a single number hides.

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot([group_A_times, group_B_times])
plt.xticks([1, 2], ["Variant A", "Variant B"])
plt.ylabel("time on page (minutes)")
plt.title("Two groups: are the averages really different?")
plt.show()

### Step 5 - run the t-test

One function does it. We pass `equal_var=False` (this is called Welch's t-test), the
safer default because it does not assume the two groups have exactly the same spread.

In [ ]:
# ttest_ind compares the averages of two independent groups.
# It returns the test statistic and the p-value.
t_statistic, p_value = stats.ttest_ind(group_A_times, group_B_times, equal_var=False)

print("t-statistic:", round(t_statistic, 3))
print()
explain_p_value(p_value)

### What that means

The t-statistic is the **difference in averages measured in standard-error units**:
signal divided by noise. A big t lands far out in the tail, which gives a small
p-value. Because we built B to be genuinely higher, a small p-value here is exactly
what we expected.

## Test 2 - chi-square test (compare CONVERSION RATES)

Use this when your metric is a **yes/no event counted across users**, such as 'did the
user finish paying?'. This is the classic A/B-test metric. We compare the conversion
rate of A with the conversion rate of B.

### Step 6 - make conversion counts for two variants

For each variant we record how many users converted (finished paying) and how many did
not. We build B to convert a little more often.

In [ ]:
# Variant A: 1000 users, about 11% convert.
users_A = 1000
converted_A = 110
not_converted_A = users_A - converted_A

# Variant B: 1000 users, about 13.8% convert.
users_B = 1000
converted_B = 138
not_converted_B = users_B - converted_B

# Conversion rates as percentages.
print("Variant A conversion rate:", round(converted_A / users_A * 100, 1), "%")
print("Variant B conversion rate:", round(converted_B / users_B * 100, 1), "%")

### Step 7 - build the 2x2 table

The chi-square test reads a small table of counts: one row per variant, one column for
'converted' and one for 'did not convert'.

In [ ]:
# Each row is one variant: [converted, not converted].
counts_table = np.array([
    [converted_A, not_converted_A],   # Variant A
    [converted_B, not_converted_B],   # Variant B
])

print("The 2x2 table of counts:")
print(counts_table)

### Step 8 - run the chi-square test

`chi2_contingency` compares the counts we actually saw against the counts we would
expect if conversion did NOT depend on the variant.

In [ ]:
# This returns four things; we only need the statistic and the p-value here.
chi2_statistic, p_value, degrees_of_freedom, expected_counts = stats.chi2_contingency(counts_table)

print("chi-square statistic:", round(chi2_statistic, 3))
print()
explain_p_value(p_value)

### What that means

A large chi-square statistic means the counts we observed are far from what we would
expect under 'the variant makes no difference'. This is the workhorse test for the
everyday conversion-rate A/B test, and it is exactly what notebook 2 uses on a full
experiment.

## Test 3 - one-way ANOVA (compare 3 or more groups)

When you test three or more designs at once (A/B/C), do NOT run a pile of separate
t-tests, because each extra test is another chance for a false alarm. ANOVA answers
the combined question with a single test: **do any of the group averages differ?**

### Step 9 - make data for three groups

Three onboarding flows for new users. We build group C to be a bit higher than A and
B, so at least one group genuinely differs. The metric is a satisfaction score out of
ten-ish.

In [ ]:
# 80 users per group, a numeric satisfaction score.
group_A_scores = np.random.normal(7.0, 1.2, size=80)
group_B_scores = np.random.normal(7.1, 1.2, size=80)
group_C_scores = np.random.normal(7.9, 1.2, size=80)

print("group A average:", round(group_A_scores.mean(), 2))
print("group B average:", round(group_B_scores.mean(), 2))
print("group C average:", round(group_C_scores.mean(), 2))

### Step 10 - look at the three groups first

Same habit as before: draw them side by side and look before testing.

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot([group_A_scores, group_B_scores, group_C_scores])
plt.xticks([1, 2, 3], ["Flow A", "Flow B", "Flow C"])
plt.ylabel("satisfaction score")
plt.title("Three groups: does any one of them differ?")
plt.show()

### Step 11 - run the one-way ANOVA

`f_oneway` takes the groups and returns an F-statistic and a p-value.

In [ ]:
# Pass all three groups. ANOVA compares the spread BETWEEN the group averages
# against the random spread WITHIN each group.
f_statistic, p_value = stats.f_oneway(group_A_scores, group_B_scores, group_C_scores)

print("F-statistic:", round(f_statistic, 3))
print()
explain_p_value(p_value)

### What that means

A small p-value here says *at least one* group differs from the others, but NOT which
one. To find which, you would follow up with post-hoc pairwise comparisons (for
example Tukey's test) that correct for the extra comparisons you are making.

### Stretch (optional) - a non-parametric test for messy data

Skip this if you are new to statistics. The t-test assumes each group is roughly a
bell curve. Real industry metrics are often not: revenue per user is a classic
offender, where most users spend little and a handful spend a fortune, giving a long
right tail. When the data is that skewed, we reach for a **non-parametric** test that
works on ranks instead of raw values, so one giant spender cannot dominate.

`mannwhitneyu` is the ranks-based cousin of the two-sample t-test. Below we make a
skewed revenue metric and compare two variants with it.

In [ ]:
# A skewed "revenue per user" metric: most users small, a few very large.
# The exponential distribution has exactly this long right tail.
revenue_A = np.random.exponential(scale=200.0, size=300)          # average around Rs 200
revenue_B = np.random.exponential(scale=230.0, size=300)          # B a bit higher

print("Variant A median revenue:", round(np.median(revenue_A), 1))
print("Variant B median revenue:", round(np.median(revenue_B), 1))
print()

# Mann-Whitney U compares the two groups using ranks, not the raw rupee amounts.
u_statistic, p_value = stats.mannwhitneyu(revenue_A, revenue_B, alternative="two-sided")
print("Mann-Whitney U statistic:", round(u_statistic, 1))
explain_p_value(p_value)

## What you just did

You ran the three core tests, each in a single line of `scipy.stats`:

- **t-test** compares two **averages** (a numeric metric like time on page).
- **chi-square** compares two **conversion rates** (a yes/no metric).
- **ANOVA** compares **three or more** group averages at once.

And in the stretch you met a non-parametric test for skewed metrics. Picking the right
test for your data is the real skill; the library does the arithmetic. Keep the rule
we repeated: a small p-value is evidence against the null, NOT the probability that
the null is true.

Next notebook: `02_did_the_change_help.ipynb`, where we take one full checkout A/B test
from raw data to a decision a product manager could act on.